# Clinical Biomedical NER & Hybrid RAG Knowledge Base (`PubMedBERT` + `ChromaDB` + `BM25` + `CrossEncoder`)
Downloads `GBaker/MedQA-USMLE-4-options` and `qiaojin/PubMedQA`, saves the Hugging Face `d4data/biomedical-ner-all` pipeline to `./biomedical_ner_model`, and builds the `ProductionClinicalHybridRAG` index.

In [ ]:
# !pip install -q datasets transformers sentence-transformers chromadb rank-bm25 torch numpy

In [2]:
from __future__ import annotations
import os
import re
from typing import Any, Dict, List, Tuple
import numpy as np
import torch
import chromadb
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder

/Users/yash/Library/A-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1104 22:54:57.493000 50843 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Load Real Clinical Vignettes (`MedQA-USMLE`) & Biomedical Abstracts (`PubMedQA`)

In [3]:
def load_clinical_notes_and_pubmedqa(n_notes: int = 30, n_pubmed: int = 300) -> Tuple[List[str], List[Dict[str, str]]]:
    print("Downloading real MedQA-USMLE clinical vignettes & PubMedQA abstracts...")
    medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split=f"train[:{n_notes}]")
    clinical_notes = [row["question"] for row in medqa]

    pubmed = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split=f"train[:{n_pubmed}]")
    corpus = []
    for i, row in enumerate(pubmed):
        ctx_list = row["context"]["contexts"] if isinstance(row["context"], dict) else [str(row["context"])]
        abstract_text = " ".join(ctx_list) + " Conclusion: " + str(row.get("long_answer", ""))
        corpus.append({
            "doc_id": f"PUBMED-{row.get('pubid', 10000 + i)}",
            "title": str(row["question"]),
            "source": f"PubMed ID {row.get('pubid', i)} ({row.get('final_decision', 'clinical')})",
            "text": abstract_text[:1500],
        })
    print(f"Loaded {len(clinical_notes)} real USMLE patient vignettes & {len(corpus)} real PubMedQA abstracts.")
    return clinical_notes, corpus

usmle_notes, pubmed_corpus = load_clinical_notes_and_pubmedqa(n_notes=30, n_pubmed=300)

Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 24687.19 examples/s]

Loaded 30 real USMLE patient vignettes & 300 real PubMedQA abstracts.


## Initialize & Save Biomedical NER Pipeline (`./biomedical_ner_model`)

In [4]:
class HuggingFaceClinicalNLP:
    """Biomedical Transformer NER + NegEx Clinical Scope Analyzer."""

    NEGATION_PATTERNS = ("denies", "denied", "no evidence of", "without", "negative for", "unremarkable for")

    def __init__(self, model_name: str = "d4data/biomedical-ner-all"):
        device_id = 0 if torch.cuda.is_available() else -1
        print(f"Loading Hugging Face Biomedical NER model: {model_name}...")
        self.ner_pipe = hf_pipeline(
            "ner",
            model=model_name,
            aggregation_strategy="simple",
            device=device_id,
        )

    def extract_clinical_entities(self, note_text: str) -> Dict[str, Any]:
        raw_entities = self.ner_pipe(note_text)
        affirmed_problems: List[str] = []
        negated_problems: List[str] = []
        medications: List[str] = []
        diagnostic_procedures: List[str] = []

        note_lower = note_text.lower()
        for ent in raw_entities:
            word = ent["word"].replace("##", "").strip()
            if len(word) < 3:
                continue
            label = ent["entity_group"].upper()
            start = int(ent.get("start", 0))
            context_window = note_lower[max(0, start - 45) : start]
            is_negated = any(pat in context_window for pat in self.NEGATION_PATTERNS)

            if label in {"SIGN_SYMPTOM", "DISEASE_DISORDER"}:
                if is_negated:
                    negated_problems.append(word)
                else:
                    affirmed_problems.append(word)
            elif label in {"MEDICATION", "THERAPEUTIC_PROCEDURE"}:
                medications.append(word)
            elif label in {"DIAGNOSTIC_PROCEDURE", "LAB_VALUE"}:
                diagnostic_procedures.append(word)

        allergies = [
            alg.capitalize()
            for alg in ("penicillin", "sulfa", "cephalosporin", "aspirin", "vancomycin")
            if alg in note_lower and ("allerg" in note_lower or "anaphylaxis" in note_lower)
        ]

        return {
            "affirmed_clinical_findings": sorted(list(set(affirmed_problems))),
            "negated_clinical_findings": sorted(list(set(negated_problems))),
            "extracted_medications": sorted(list(set(medications))),
            "diagnostic_procedures": sorted(list(set(diagnostic_procedures))),
            "documented_allergies": allergies if allergies else ["NKDA"],
        }

nlp_engine = HuggingFaceClinicalNLP()
print("Sample extracted entities:", nlp_engine.extract_clinical_entities(usmle_notes[0]))

save_path = "biomedical_ner_model"
os.makedirs(save_path, exist_ok=True)
nlp_engine.ner_pipe.model.save_pretrained(save_path)
nlp_engine.ner_pipe.tokenizer.save_pretrained(save_path)
print(f"Biomedical NER model saved successfully at: {save_path}")

Loading Hugging Face Biomedical NER model: d4data/biomedical-ner-all...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13605.31it/s]


Sample extracted entities: {'affirmed_clinical_findings': ['cost', 'feels well', 'tebral', 'tenderness'], 'negated_clinical_findings': [], 'extracted_medications': ['cranberry'], 'diagnostic_procedures': ['. 5', '122 / 77 mm', '80 /', '97. 7°f', 'blood pressure', 'oxygen saturation', 'physical exam', 'pulse', 'respiration', 'temperature', 'worse'], 'documented_allergies': ['NKDA']}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s]

Biomedical NER model saved successfully at: biomedical_ner_model


## Build & Test Hybrid Clinical RAG (`ChromaDB` + `PubMedBERT` + `BM25Okapi` + `CrossEncoder`)

In [5]:
class ProductionClinicalHybridRAG:
    """ChromaDB Dense Vector Store + BM25Okapi Lexical Index + Cross-Encoder Reranker."""

    def __init__(
        self,
        corpus: List[Dict[str, str]],
        embed_model_name: str = "NeuML/pubmedbert-base-embeddings",
        reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        self.corpus = corpus
        print(f"Loading PubMedBERT Embedder ({embed_model_name}) & CrossEncoder Reranker...")
        self.embedder = SentenceTransformer(embed_model_name)
        self.reranker = CrossEncoder(reranker_model_name)

        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.get_or_create_collection(
            name="pubmedqa_clinical_guidelines",
            metadata={"hnsw:space": "cosine"},
        )

        texts = [f"{doc['title']}. {doc['text']}" for doc in self.corpus]
        ids = [doc["doc_id"] for doc in self.corpus]
        embeddings = self.embedder.encode(texts, show_progress_bar=False, normalize_embeddings=True).tolist()

        self.collection.upsert(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=[{"title": d["title"], "source": d["source"]} for d in self.corpus],
        )

        tokenized_corpus = [re.findall(r"[a-z0-9]+", t.lower()) for t in texts]
        self.bm25 = BM25Okapi(tokenized_corpus)
        self.id_to_idx = {doc["doc_id"]: idx for idx, doc in enumerate(self.corpus)}
        print(f"Indexed {len(self.corpus)} biomedical documents into ChromaDB + BM25Okapi.")

    def hybrid_search_with_rerank(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        q_emb = self.embedder.encode([query], normalize_embeddings=True).tolist()
        dense_res = self.collection.query(query_embeddings=q_emb, n_results=min(10, len(self.corpus)))
        dense_ids = dense_res["ids"][0]

        q_tokens = re.findall(r"[a-z0-9]+", query.lower())
        bm25_scores = self.bm25.get_scores(q_tokens)
        bm25_top_indices = np.argsort(-bm25_scores)[:10]
        bm25_ids = [self.corpus[int(i)]["doc_id"] for i in bm25_top_indices]

        rrf_scores: Dict[str, float] = {}
        for rank, doc_id in enumerate(dense_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (60.0 + rank + 1)
        for rank, doc_id in enumerate(bm25_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (60.0 + rank + 1)

        candidate_ids = sorted(rrf_scores.keys(), key=lambda d: rrf_scores[d], reverse=True)[:8]
        candidates = [self.corpus[self.id_to_idx[cid]] for cid in candidate_ids]

        pairs = [[query, f"{c['title']}. {c['text']}"] for c in candidates]
        ce_scores = self.reranker.predict(pairs)
        ranked_order = np.argsort(-np.asarray(ce_scores))[:top_k]

        results = []
        for idx in ranked_order:
            doc = candidates[int(idx)]
            results.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "source": doc["source"],
                "snippet": doc["text"][:450] + "...",
                "cross_encoder_score": round(float(ce_scores[int(idx)]), 4),
            })
        return results

rag_engine = ProductionClinicalHybridRAG(corpus=pubmed_corpus)
print("Sample RAG search:", rag_engine.hybrid_search_with_rerank("pneumonia antibiotic penicillin allergy", top_k=2))

Loading PubMedBERT Embedder (NeuML/pubmedbert-base-embeddings) & CrossEncoder Reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7851.56it/s]


Indexed 300 biomedical documents into ChromaDB + BM25Okapi.
Sample RAG search: [{'doc_id': 'PUBMED-17562682', 'title': 'Is Panton-Valentine leucocidin associated with the pathogenesis of Staphylococcus aureus bacteraemia in the UK?', 'source': 'PubMed ID 17562682 (maybe)', 'snippet': 'The morbidity and mortality associated with Panton-Valentine leucocidin (PVL)-positive Staphylococcus aureus suggest that this toxin is a key marker of disease severity. Nevertheless, the importance of PVL in the pathogenesis of primary bacteraemia caused by S. aureus is uncertain. We have determined the prevalence of PVL-encoding genes among isolates of S. aureus from bacteraemic patients. Consecutive bacteraemia isolates of S. aureus (n=244) fr...', 'cross_encoder_score': -5.0376}, {'doc_id': 'PUBMED-27287237', 'title': 'Assessment of appropriate antimicrobial prescribing: do experts agree?', 'source': 'PubMed ID 27287237 (yes)', 'snippet': "Little is known about the validity and reliability of expert a